In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
dataset["train"] = dataset["train"].select(range(1000))
dataset["test"] = dataset["test"].select(range(250))
print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
from transformers import TrainingArguments

training_args = TrainingArguments(
        output_dir=' /results', eval_strategy="epoch", learning_rate=2e-5, per_device_train_batch_size=16,
        per_device_eval_batch_size=16, num_train_epochs=3, weight_decay=0.01, logging_dir='./logs', logging_steps=100
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [5]:
from transformers import Trainer

trainer = Trainer(
        model=model, args=training_args, train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"]
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.001183
2,0.018117,0.000512
3,0.018117,0.000409


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=0.009871474097645471, metrics={'train_runtime': 285.5208, 'train_samples_per_second': 10.507, 'train_steps_per_second': 0.662, 'total_flos': 789333166080000.0, 'train_loss': 0.009871474097645471, 'epoch': 3.0})

In [6]:
results = trainer.evaluate()
print(f"Evaluation results: {results}")

Training Loss,Validation Loss,Epoch
0.018117,0.000409,3


Evaluation results: {'eval_loss': 0.00040885453927330673}


In [8]:
import torch


def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # only if you have GPU
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()
    return "positive" if predicted_class == 1 else "negative"


sample_text = "I absolutely loved this movie!"
print(predict_sentiment(sample_text))

negative


In [9]:
model.save_pretrained("./fine-tuned-model")
tokenizer.save_pretrained("./fine-tuned-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine-tuned-model/tokenizer_config.json',
 './fine-tuned-model/tokenizer.json')

In [11]:
from transformers import BertForSequenceClassification, BertTokenizer

model = BertForSequenceClassification.from_pretrained("./fine-tuned-model")
tokenizer = BertTokenizer.from_pretrained("./fine-tuned-model")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]